# Module 05 — Loss Functions, Optimizers & the Training Loop

**Prerequisites:** Module 04 (Building Neural Networks: From Scratch to `nn.Module`)
**Time:** ~60 minutes

## Learning Objectives

- Choose the correct loss function for a regression vs. a classification task.
- Explain what `torch.optim` optimizers do, and how they use `.grad`.
- Write the canonical 5-step PyTorch training loop from memory, and explain every line.
- Diagnose the most common "my model isn't learning" bugs related to this loop.


In [1]:
import torch
import torch.nn as nn


## Loss Functions: Measuring "How Wrong"

A loss function takes the model's prediction and the true target, and returns a single number: how bad the prediction is. Lower is better; a perfect prediction gives a loss of 0.

| Loss | Task | What it expects |
|---|---|---|
| `nn.MSELoss()` | Regression (predicting a continuous number) | Predictions and targets, any matching shape |
| `nn.CrossEntropyLoss()` | Multi-class classification | **Raw logits** (not softmax!) and integer class labels |
| `nn.BCEWithLogitsLoss()` | Binary classification | **Raw logits** (not sigmoid!) and 0/1 float targets |

### Mean Squared Error (Regression)

MSE computes the average of `(prediction - target)^2` across all examples. Squaring makes all errors positive and penalizes large errors more heavily than small ones.


In [2]:
predictions = torch.tensor([2.5, 0.0, 2.1])
targets     = torch.tensor([3.0, -0.5, 2.0])

mse = nn.MSELoss()
loss = mse(predictions, targets)
print("MSE loss:", loss.item())

# Manually verifying the formula: mean of (pred - target)^2
manual = ((predictions - targets) ** 2).mean()
print("manual   :", manual.item())


MSE loss: 0.17000000178813934
manual   : 0.17000000178813934


### Cross-Entropy (Multi-class Classification)

This is the one that trips up nearly every beginner, so let's be explicit: `nn.CrossEntropyLoss` expects **raw, unnormalized scores** from the model (called "logits") — *not* probabilities you've already run through softmax. It applies `log_softmax` and negative-log-likelihood *internally*, in one numerically stable step.


In [3]:
# 3 samples, 4 possible classes each -- these are RAW LOGITS, not probabilities
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.0],   # model is fairly confident about class 0
    [0.1, 0.2, 3.0, 0.1],   # model is fairly confident about class 2
    [1.0, 1.0, 1.0, 1.0],   # model has no idea (all equal)
])
true_classes = torch.tensor([0, 2, 1])   # the correct class index for each sample -- NOT one-hot!

ce = nn.CrossEntropyLoss()
loss = ce(logits, true_classes)
print("cross-entropy loss:", loss.item())


cross-entropy loss: 0.6821635365486145


Two details worth memorizing, because getting either wrong produces a model that trains "successfully" but learns the wrong thing:

1. **Don't apply softmax yourself** before passing predictions into `nn.CrossEntropyLoss`. If you do, you're applying softmax *twice* (once manually, once inside the loss function), which silently produces incorrect gradients — the model may still appear to train, just worse than it should.
2. **Targets are integer class indices** (e.g., `2` meaning "class 2"), not one-hot vectors (`[0,0,1,0]`). This is different from some other frameworks.

If you need actual probabilities for reporting/inspection (not for computing loss), apply `torch.softmax(logits, dim=-1)` separately, purely for display.


In [4]:
probabilities = torch.softmax(logits, dim=-1)
print(probabilities)
print("each row sums to 1.0:", probabilities.sum(dim=-1))


tensor([[0.6050, 0.2226, 0.0905, 0.0819],
        [0.0470, 0.0519, 0.8541, 0.0470],
        [0.2500, 0.2500, 0.2500, 0.2500]])
each row sums to 1.0: tensor([1.0000, 1.0000, 1.0000])


### Binary Cross-Entropy (Binary Classification)

For a yes/no, 0/1 classification task, `nn.BCEWithLogitsLoss` is the standard choice — again, it expects raw logits (a single number per sample, positive meaning "leans toward class 1"), and applies sigmoid internally.


In [5]:
logits_binary = torch.tensor([2.0, -1.0, 0.1])   # raw scores, one per sample
targets_binary = torch.tensor([1.0, 0.0, 1.0])     # 0 or 1, as floats

bce = nn.BCEWithLogitsLoss()
loss = bce(logits_binary, targets_binary)
print("binary cross-entropy loss:", loss.item())


binary cross-entropy loss: 0.36152878403663635


## Optimizers: Automating the Update Step

Recall Module 04's manual update:

```python
with torch.no_grad():
    W -= learning_rate * W.grad
    b -= learning_rate * b.grad
W.grad.zero_()
b.grad.zero_()
```

An **optimizer** wraps exactly this logic (and, for more advanced optimizers, quite a bit more bookkeeping) behind two method calls: `optimizer.step()` (apply the update) and `optimizer.zero_grad()` (reset gradients).


In [6]:
model = nn.Linear(1, 1)

# SGD = Stochastic Gradient Descent -- the update we wrote by hand in Module 04
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# model.parameters() hands the optimizer every registered parameter (weight + bias here),
# which is exactly why parameter registration in nn.Module (Module 04) matters.
print("Optimizer is tracking these parameter shapes:")
for p in model.parameters():
    print(" ", tuple(p.shape))


Optimizer is tracking these parameter shapes:
  (1, 1)
  (1,)


| Optimizer | Typical use | Typical starting LR |
|---|---|---|
| `SGD(params, lr, momentum=...)` | Simple baselines, well-tuned CNN pipelines | 0.01–0.1 |
| `Adam(params, lr)` | Good general-purpose default | 1e-3 |
| `AdamW(params, lr, weight_decay=...)` | Fine-tuning, transformer-style models | 1e-4–1e-3 |

`Adam` is the most common starting point in practice — it adapts its effective step size per-parameter, which usually makes it more forgiving of an imperfectly-chosen learning rate than plain `SGD`.

**A note on learning rate:** too large, and the loss can oscillate wildly or diverge (grow rather than shrink); too small, and training can be painfully slow, or get stuck in a mediocre solution. We'll deliberately trigger both problems in the debugging challenge below so you can *recognize* them from the printed loss curve alone.


## The Canonical Training Loop

Here is the pattern used in essentially every PyTorch project you will ever encounter. Memorize the five steps inside the inner loop; they don't change, regardless of model architecture or task.


In [7]:
torch.manual_seed(0)

# Synthetic regression data again
x = torch.linspace(-5, 5, 200).unsqueeze(1)
y = 3 * x + 2 + torch.randn_like(x) * 0.5

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

n_epochs = 100
for epoch in range(n_epochs):
    # 1. Reset gradients from the previous iteration (Module 03: they accumulate otherwise)
    optimizer.zero_grad()

    # 2. Forward pass: compute predictions
    predictions = model(x)

    # 3. Compute the loss: how wrong are the predictions?
    loss = loss_fn(predictions, y)

    # 4. Backward pass: compute d(loss)/d(every parameter), via autograd
    loss.backward()

    # 5. Update parameters using the computed gradients
    optimizer.step()

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")

print(f"\nlearned weight: {model.weight.item():.3f} (true=3.0)")
print(f"learned bias:   {model.bias.item():.3f} (true=2.0)")


epoch   0 | loss 125.9334
epoch  20 | loss 3.1199
epoch  40 | loss 1.4850
epoch  60 | loss 0.7897
epoch  80 | loss 0.4798

learned weight: 2.984 (true=3.0)
learned bias:   1.655 (true=2.0)


```text
Data
 |
 v
Model            <- forward pass (step 2)
 |
 v
Prediction
 |
 v
Loss             <- compare to target (step 3)
 |
 v
Backward pass    <- loss.backward() (step 4)
 |
 v
Gradients        <- stored in param.grad
 |
 v
Optimizer        <- optimizer.step() (step 5)
 |
 v
Updated parameters
 |
 v
Repeat
```

Every line of this loop connects directly to something you've already built by hand: `zero_grad()` replaces the manual `.grad.zero_()` calls, `loss.backward()` is exactly the autograd call from Module 03, and `optimizer.step()` replaces the manual `with torch.no_grad(): param -= lr * param.grad` update from Module 04.


## 🐛 Debugging Challenge: Forgetting `zero_grad()`

Predict what happens to the loss curve if we drop `optimizer.zero_grad()` from the loop. Then run it and see.


In [8]:
torch.manual_seed(0)
model_bug = nn.Linear(1, 1)
optimizer_bug = torch.optim.SGD(model_bug.parameters(), lr=0.01)

for epoch in range(100):
    # optimizer_bug.zero_grad()   <-- MISSING ON PURPOSE
    predictions = model_bug(x)
    loss = loss_fn(predictions, y)
    loss.backward()
    optimizer_bug.step()
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")


epoch   0 | loss 77.6625
epoch  20 | loss 28.5690
epoch  40 | loss 22.7637
epoch  60 | loss 78.3909
epoch  80 | loss 6.4786


Depending on the exact setup, you'll typically see the loss decrease far more slowly, plateau at a worse value, or behave erratically — because gradients from every past epoch are piling on top of the current epoch's real gradient, corrupting the update direction and magnitude. This is a bug that **does not crash** — it just quietly makes training worse, which is exactly what makes it dangerous. Whenever a model trains "worse than it should" with no error message, missing `zero_grad()` is one of the first things to check.


## 🐛 Debugging Challenge: Learning Rate Too High

Now let's see what a learning rate that's far too large looks like.


In [9]:
torch.manual_seed(0)
model_diverge = nn.Linear(1, 1)
optimizer_diverge = torch.optim.SGD(model_diverge.parameters(), lr=5.0)   # WAY too high

for epoch in range(10):
    optimizer_diverge.zero_grad()
    predictions = model_diverge(x)
    loss = loss_fn(predictions, y)
    loss.backward()
    optimizer_diverge.step()
    print(f"epoch {epoch} | loss {loss.item():.4f}")


epoch 0 | loss 77.6625
epoch 1 | loss 521209.0000
epoch 2 | loss 3604237568.0000
epoch 3 | loss 24931823779840.0000
epoch 4 | loss 172463174820298752.0000
epoch 5 | loss 1192995097041143070720.0000
epoch 6 | loss 8252412674555513669681152.0000
epoch 7 | loss 57085174970747885972331954176.0000
epoch 8 | loss 394880512240076715511605262024704.0000
epoch 9 | loss inf


If the loss is growing (or oscillating wildly, or becomes `nan`/`inf`) rather than shrinking, that's the signature of a learning rate that's too high: each update overshoots past the minimum, landing further away than where it started. The fix, in practice, is almost always simply to try a smaller learning rate (often by 10x) and see if the loss starts decreasing smoothly.


## Exercises

🟢 **Beginner:** Using the `logits` and `true_classes` tensors from earlier in this notebook, compute the predicted class for each sample using `logits.argmax(dim=-1)`, and compare against `true_classes` to compute accuracy (fraction correct).

🟡 **Intermediate:** Repeat the training loop above, but replace `SGD` with `torch.optim.Adam(model.parameters(), lr=0.1)`. Compare how many epochs it takes to reach a similar loss value compared to SGD with `lr=0.01`.

🔴 **Challenge:** Build a full training loop for a *binary classification* toy problem: generate `x = torch.randn(200, 2)` and `y = (x[:, 0] + x[:, 1] > 0).float()` (label 1 if the two features sum to something positive). Build a small `nn.Sequential` model (`nn.Linear(2, 8)`, `nn.ReLU()`, `nn.Linear(8, 1)`), train it with `nn.BCEWithLogitsLoss()` and `Adam`, and print the final training accuracy (remember: outputs are logits, so use `(logits > 0)` as the predicted class, since that's equivalent to `sigmoid(logits) > 0.5`).


In [10]:
# Space for your exercise solutions



## Common Mistakes

- **Applying softmax/sigmoid before `nn.CrossEntropyLoss`/`nn.BCEWithLogitsLoss`** — both loss functions expect raw logits and apply the normalization internally; doing it twice silently breaks gradients.
- **Forgetting `optimizer.zero_grad()`** — trains, but poorly, with no error to signal the problem.
- **Learning rate too high** — loss grows or oscillates instead of shrinking; try reducing it by 10x.
- **Learning rate too low** — loss shrinks agonizingly slowly, or appears to plateau early; try increasing it.
- **Wrong target format** — `CrossEntropyLoss` wants integer class indices, not one-hot vectors; `MSELoss` and `BCEWithLogitsLoss` want matching-shape float tensors.

## Mental Model

The training loop is a feedback cycle, repeated many times: *guess* (forward pass) → *grade the guess* (loss) → *figure out how to improve* (backward pass) → *actually improve* (optimizer step) → *forget the old feedback and guess again* (zero_grad). Every single training loop you will ever write, from a two-parameter linear regression to a billion-parameter language model, is this same five-step cycle repeated many, many times.

## Key Takeaways

- Choose your loss function based on the task: `MSELoss` for regression, `CrossEntropyLoss` for multi-class, `BCEWithLogitsLoss` for binary — and always pass raw logits to the classification losses.
- Optimizers (`SGD`, `Adam`, `AdamW`, ...) automate the manual gradient-descent update step from Module 04.
- The five-step training loop — zero_grad, forward, loss, backward, step — is universal across PyTorch projects.
- A loss that doesn't decrease, or decreases strangely, usually traces back to one of: missing `zero_grad()`, a learning rate that's too high or too low, or a mismatched loss function / target format.

## What's Next

**Module 06 — Datasets & DataLoaders** replaces the toy in-memory tensors used in this notebook's examples with PyTorch's `Dataset` and `DataLoader` abstractions, which is how real training data — too large to fit conveniently in one tensor — gets fed into this exact training loop, batch by batch.

## Checklist

- [ ] I can choose the correct loss function for regression vs. multi-class vs. binary classification
- [ ] I understand why classification losses expect raw logits, not softmax/sigmoid output
- [ ] I can write the 5-step training loop from memory and explain each line
- [ ] I can recognize the symptoms of a missing `zero_grad()` and a badly-chosen learning rate
